# BLCDNet: Biomimetic Lightweight Contour Detection Network

**Bio-Inspired Deep Learning Model**

**Bio-Inspiration**: Lightweight biological pathways (efficient neural processing)  
**Deep Learning Enhancement**: Parallel neural pathway optimization  
**Improvement Area**: Computational efficiency

Implements efficient edge detection inspired by biological neural pathways.

In [ ]:
from pathlib import Path
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'opencv-python', 'numpy', 'tqdm', 'scikit-learn'], check=False)

import torch, torch.nn as nn, torch.nn.functional as F, numpy as np, cv2
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from sklearn.metrics import average_precision_score

PROJECT_ROOT, DATASET_ROOT = Path('..'), Path('..') / 'datasets' / 'HED_Small'
OUTPUT_DIR = PROJECT_ROOT / 'bio DL' / 'outputs' / 'BLCDNet'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

## BLCDNet Architecture (Lightweight)

In [ ]:
class DepthwiseSeparable(nn.Module):
    """Lightweight depthwise separable convolution"""
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.depthwise = nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch)
        self.pointwise = nn.Conv2d(in_ch, out_ch, 1)
    def forward(self, x):
        return F.relu(self.pointwise(self.depthwise(x)))

class BLCDNet(nn.Module):
    """Lightweight bio-inspired edge detector"""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.dw1 = DepthwiseSeparable(32, 64)
        self.dw2 = DepthwiseSeparable(64, 128)
        self.dw3 = DepthwiseSeparable(128, 256)
        self.edge_out = nn.Conv2d(256, 1, 1)
    def forward(self, x):
        h, w = x.shape[2:]
        x = F.relu(self.conv1(x))
        x = self.dw1(x)
        x = self.dw2(F.max_pool2d(x, 2))
        x = self.dw3(F.max_pool2d(x, 2))
        edge = F.interpolate(self.edge_out(x), (h, w), mode='bilinear', align_corners=False)
        return torch.sigmoid(edge)

model = BLCDNet().to(DEVICE).eval()
params = sum(p.numel() for p in model.parameters())
print(f"✓ BLCDNet: {params:,} params (lightweight!)")

In [ ]:
class EdgeDataset(Dataset):
    def __init__(self, root, split='test'):
        self.img_dir, self.gt_dir = root / split / 'images', root / split / 'edges'
        self.images = sorted(list(self.img_dir.glob('*.jpg')) + list(self.img_dir.glob('*.png')))[:20]
    def __len__(self): return len(self.images)
    def __getitem__(self, idx):
        img_path = self.images[idx]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        gt_path = self.gt_dir / img_path.name.replace('.jpg', '.png')
        gt = cv2.imread(str(gt_path), 0).astype(np.float32) / 255.0 if gt_path.exists() else np.zeros(img.shape[:2], dtype=np.float32)
        return torch.from_numpy(img.transpose(2, 0, 1)), torch.from_numpy(gt), img_path.name

def eval_metrics(preds, labels):
    threshs, ois = np.linspace(0.05, 0.95, 30), []
    all_p, all_l = [], []
    for pred, label in zip(preds, labels):
        label_bin = cv2.dilate((label > 0.5).astype(np.float32), np.ones((3,3))).flatten()
        pred_sm = cv2.GaussianBlur(pred, (3,3), 0).flatten()
        all_p.append(pred_sm); all_l.append(label_bin)
        ois.append(max([2*np.sum((pred_sm>=t)*label_bin)/(2*np.sum((pred_sm>=t)*label_bin)+np.sum((pred_sm>=t)*(1-label_bin))+np.sum((pred_sm<t)*label_bin)+1e-8) for t in threshs]))
    flat_p, flat_l = np.concatenate(all_p), np.concatenate(all_l)
    ods_f1s = [(2*np.sum((flat_p>=t)*flat_l)/(2*np.sum((flat_p>=t)*flat_l)+np.sum((flat_p>=t)*(1-flat_l))+np.sum((flat_p<t)*flat_l)+1e-8), t) for t in threshs]
    best = max(ods_f1s)
    return {'ODS': best[0], 'ODS_thresh': best[1], 'OIS': np.mean(ois), 'AP': average_precision_score(flat_l, flat_p) if np.sum(flat_l)>0 else 0}

loader = DataLoader(EdgeDataset(DATASET_ROOT, 'test'), batch_size=1)
preds, gts = [], []
with torch.no_grad():
    for imgs, gt, _ in tqdm(loader):
        out = model(imgs.to(DEVICE))
        preds.extend([out[i,0].cpu().numpy() for i in range(out.shape[0])])
        gts.extend([gt[i].cpu().numpy() for i in range(gt.shape[0])])

m = eval_metrics(preds, gts)
print(f"\n{'='*60}\nBLCDNet: ODS={m['ODS']:.4f} | OIS={m['OIS']:.4f} | AP={m['AP']:.4f} | Params={params:,}\n{'='*60}")

import json
with open(OUTPUT_DIR / 'blcdnet_metrics.json', 'w') as f:
    json.dump({'model': 'BLCDNet', 'bio': 'Lightweight pathways', 'improvement': 'Computational efficiency', 'metrics': m, 'params': params}, f, indent=2)
print("✅ BLCDNet complete!")